# Fine Tuning LLM for Basic Chat Generation

In [1]:
pip install transformers

In [2]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import transformers

# Dataset Format:
### {
  'input' : {'role': 'user' , 'mood':['flirty'  ,'friendly', 'serious'], 'text':'xxx'},
'output' : {role': 'bot' , 'mood':['flirty'  ,'friendly', 'serious'], 'text':'xxx'}
}

In [4]:
from google.colab import files
data = files.upload()

Saving chat_data.jsonl to chat_data.jsonl


In [31]:
from datasets import load_dataset
dataset = load_dataset("text", data_files="chat_data.jsonl", split ="train")
print(dataset)
print(dataset[:2])

Dataset({
    features: ['text'],
    num_rows: 166
})
{'text': ['{"role": "user", "text": "Hello dear! How are you?"}', '{"role": "bot", "text": "Hey darling!! I am good, what about you? How was your day dear?"}']}


In [32]:
import json

parsed = [json.loads(x) for x in dataset["text"]]

pairs = []

for i in range(len(parsed) - 1):
    if parsed[i]["role"] == "user" and parsed[i + 1]["role"] == "bot":
        pairs.append({
            "prompt": parsed[i]["text"],
            "completion": parsed[i + 1]["text"]
        })

pairs[:2]

[{'prompt': 'Hello dear! How are you?',
  'completion': 'Hey darling!! I am good, what about you? How was your day dear?'},
 {'prompt': 'Hello dear! How are you?',
  'completion': 'Hey darling!! I am good, what about you? How was your day dear?'}]

In [33]:
from datasets import Dataset
dataset = Dataset.from_list(pairs)
dataset

Dataset({
    features: ['prompt', 'completion'],
    num_rows: 83
})

In [34]:

len(dataset)

83

In [35]:
dataset[0]

{'prompt': 'Hello dear! How are you?',
 'completion': 'Hey darling!! I am good, what about you? How was your day dear?'}

In [19]:
from transformers import AutoTokenizer , AutoModelForCausalLM

In [20]:
model_name = "Qwen/Qwen2.5-1.5B"

In [21]:
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
from transformers import pipeline
llm = pipeline(model=model_name)
llm("Hey dear, how are you?")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Hey dear, how are you? I am happy to see you! But first, I want to ask you a question. Have you ever been to a town called "Brussels" before? It\'s a place in Belgium, where people speak French and the streets are very narrow. The weather there is usually cold and rainy in winter, and it\'s warm and sunny in summer. There are many buildings, bridges, and trees in Brussels, and it\'s a beautiful city. Do you want to visit it yourself?\n\nI\'m sorry, but I\'m afraid I have a question of my own. Can you tell me what the capital of the United States is? And also, what is the population of the largest city in the United States?'}]

In [49]:
llm("Do you like cricket? 🙂")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Do you like cricket? 🙂\nIn cricket, the batsman hits the ball with his bat and tries to score runs. It’s very interesting to watch the game. You have to be quick to react. But if you have a problem, you can use your hands to help you. That’s why you need a bat that is easy to hold.\nFor the best bats, the players need to be very careful. They need to choose the right bat for themselves. That’s why the players practice hitting the ball and using their bats.\nThe batsman does not have to hit the ball again and again. He can run around the field, but the ball has to hit something. It can hit the ground, the batsman or the third player. But the batsman can’t run to the ground, the batsman or the third player. The batsman can run to the ground or the batsman, but not to the third player. It’s important to remember that the batsman can’t run to the ground or the batsman. He can run to the ground, the batsman or the third player. But he can’t run to the ground or the bats

In [36]:
tokenizer.pad_token = tokenizer.eos_token
def tokenize(data):
  prompt = data['prompt'] + "\n" + data['completion']
  tokenized = tokenizer(
      prompt,
      truncation = True,
      max_new_tokens=16
  )
  return tokenized
from transformers import DataCollatorForLanguageModeling



dataset = dataset.map(tokenize , remove_columns=dataset.column_names)
from pprint import pprint
pprint(dataset[:2])


Map:   0%|          | 0/83 [00:00<?, ? examples/s]

{'attention_mask': [[1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1],
                    [1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
                     1,
               

In [37]:
from peft import get_peft_model , LoraConfig , TaskType

config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model , config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [38]:
from transformers import TrainingArguments , Trainer

train_args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=2,

    learning_rate=2e-5,

    num_train_epochs=10,

    logging_steps=1,

    save_strategy="epoch",

    weight_decay=0.01,

    warmup_steps=5,

    fp16=True,

    report_to="none"
)

In [39]:


# 2. RE-CREATE the collator so it picks up the change
from transformers import DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 3. RE-INITIALIZE the trainer so it uses the new collator
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=dataset,
    data_collator=collator  # The refreshed collator
)

# 4. Now start
trainer.train()


Step,Training Loss
1,2.559807
2,2.505097
3,2.653969
4,2.897086
5,3.078482
6,2.605427
7,2.641967
8,2.225929
9,2.245192
10,3.002790


TrainOutput(global_step=420, training_loss=2.664690706275758, metrics={'train_runtime': 93.7761, 'train_samples_per_second': 8.851, 'train_steps_per_second': 4.479, 'total_flos': 207329237185536.0, 'train_loss': 2.664690706275758, 'epoch': 10.0})

In [43]:
def test_model(user_input):
  inputs = tokenizer(user_input, return_tensors="pt").to('cuda')
  output = model.generate(**inputs, max_new_tokens=32,temperature=1.0)
  print(tokenizer.decode(output[0], skip_special_tokens=True))

In [41]:
test_model("Hello dear, how are you?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Hello dear, how are you? I'm doing great, thank you! How are you? I'm doing great


In [42]:
test_model("Do you like music? What do you listen to?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Do you like music? What do you listen to? Do you like to sing? Do you like to dance? Do you like to


In [ ]:
test_model("kesi ho baby?")

In [44]:
test_model("We should really grab dinner somethimes you know 😁")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


We should really grab dinner somethimes you know 😁
I totally agree! I'm always hungry and I love a good meal. What's your favorite type of food?


In [46]:
test_model("What should i wear for tomorrow's party??")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


What should i wear for tomorrow's party?? Tomorrow's party sounds like a fun event! Here are some outfit suggestions to help you look your best:

**Formal Attire:**

* A classic white


In [47]:
test_model("Who is elon musk?")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Who is elon musk? Elon Musk is a billionaire entrepreneur, engineer, and investor who is best known for his work in the technology and energy sectors. He is the founder and CEO of


In [48]:
test_model("Do you like cricket? 🙂")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Do you like cricket? 🙂
I like cricket, but I don’t play it. I just watch it on TV. 🏟️🏏
I love watching cricket too! �
